In [ ]:
import sys
import os
import logging
import gc
import time
import torch
import warnings
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gliner import GLiNER
from gliner.data_processing.collator import DataCollator
from gliner.training import Trainer, TrainingArguments
from transformers import TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType,PeftModel

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
sys.path

/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python311.zip',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages',
 '/tmp/tmpxous61wc']

In [6]:
src_path=os.path.join(os.path.dirname(os.getcwd()),'src')
sys.path.append(src_path)
sys.path


['/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python311.zip',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages',
 '/tmp/tmpxous61wc',
 '/opt/app/notebooks/abhishek/active_gliner/src']

In [7]:
from config.settings import Settings
settings = Settings()

print(f"Settings cache_dir: {settings.cache_dir}")
print(f"Cache absolute path: {settings.cache_dir.resolve()}")
print(f"Does cache dir contain 'notebooks': {'notebooks' in str(settings.cache_dir)}")
settings

Settings cache_dir: /opt/app/notebooks/abhishek/active_gliner/cache
Cache absolute path: /opt/app/notebooks/abhishek/active_gliner/cache
Does cache dir contain 'notebooks': True


Settings(seed=42, batch_size=8, model=knowledgator/modern-gliner-bi-large-v1.0)

In [8]:
print("=== Integration Test ===")
from utils.logging import setup_logging
from utils.reproducibility import set_all_seeds
from utils.device import setup_device

# Complete setup like your original code
settings = Settings()
settings.setup()  # Apply environment and create directories

logger = setup_logging(log_dir=str(settings.logs_dir))
set_all_seeds(seed=settings.global_seed, logger=logger)
device = setup_device(logger=logger)

logger.info("All modules integrated successfully!")
print(f"Final setup: seed={settings.global_seed}, device={device}, batch_size={settings.batch_size}")



INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:ACTIVE LEARNING PIPELINE WITH PROPER TRAIN/TEST SEPARATION
INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:Log file: /opt/app/notebooks/abhishek/active_gliner/logs/ActiveLearning_20250929_123231.log
INFO:ActiveLearning:Setting all seeds to 42 for reproducibility...
INFO:ActiveLearning:Using device: cuda
INFO:ActiveLearning:CUDA version: 12.8
INFO:ActiveLearning:Number of GPUs visible: 1
INFO:ActiveLearning:Current GPU: 0
INFO:ActiveLearning:GPU Name: NVIDIA GeForce RTX 3090
INFO:ActiveLearning:GPU Memory: 23.6 GB
INFO:ActiveLearning:All modules integrated successfully!


=== Integration Test ===
Final setup: seed=42, device=cuda, batch_size=8


In [9]:
import json
from data.loader import load_mit_dataset


with open(r"../results/high_mse_2500_examples.json",mode="r") as file:
    low_n=json.load(file)


print(low_n[0])


# Load FULL test data for evaluation
test_data_path = settings.data_path / settings.test_file
labels_path = settings.data_path / settings.labels_file

if not (test_data_path.exists() and labels_path.exists()):
    raise FileNotFoundError("Test data or labels file not found!")

test_data, entity_types = load_mit_dataset(str(test_data_path), str(labels_path), "test")
logger.info(f"📊 Loaded FULL test data: {len(test_data)} examples, {len(entity_types)} entity types")


INFO:ActiveLearning:📊 Loaded FULL test data: 2442 examples, 12 entity types


{'tokenized_text': ['the', 'african', 'queen'], 'ner': [[0, 2, 'title']], 'predictions': [[2, 2, 'title']], 'scores': [0.500488817691803]}
Loading test data from: /opt/app/notebooks/abhishek/active_gliner/data/mit-movie/test.json
Processed 2442 examples
Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']


In [13]:
from generation.enc_api_label import LabelGenerator, QuotaExceededException
from dotenv import load_dotenv
load_dotenv() 


n_examples=5
label_cache = []
label_generator = LabelGenerator(model_name="qwen-3-235b-a22b-thinking-2507")
train_subset = low_n[:n_examples]

llm_labeled_data = label_generator.generate(
    low_n_examples=train_subset,
    num_samples=n_examples,
    entity_types=entity_types,
    label_cache=label_cache,
    verbose=True
)






INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:ActiveLearning:Enhanced Label Generator initialized: qwen-3-235b-a22b-thinking-2507
INFO:ActiveLearning:Cache directory: ../results/data
INFO:ActiveLearning:============================================================
INFO:ActiveLearning:ENHANCED CEREBRAS API LABEL GENERATION
INFO:ActiveLearning:============================================================
INFO:ActiveLearning:Model: qwen-3-235b-a22b-thinking-2507
INFO:ActiveLearning:Context limit: 65,536 tokens
INFO:ActiveLearning:Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']
INFO:ActiveLearning:Low confidence examples available: 5
INFO:ActiveLearning:Target labels: 5
INFO:ActiveLearning:Current cache size: 0
INFO:ActiveLearning:Rate limits: 30 req/min, 60,000 tokens/min
INFO:ActiveLearning:No existing cache found, starting fresh
INFO:ActiveLearning:Need

Converting 4 synthetic examples to NER format...
Conversion completed: 4 examples, 0 errors


In [26]:
import os
import json
import pandas as pd
from cerebras.cloud.sdk import Cerebras
from dotenv import load_dotenv

load_dotenv()
client = Cerebras(api_key=os.environ.get("CEREBRAS_API_KEY"))

# Load examples
with open("../results/high_mse_2500_examples.json", "r") as f:
    examples = json.load(f)

entity_types = ["genre", "year", "plot", "review", "song", "rating", 
                "character", "trailer", "director", "title", "actor"]

NUM_EXAMPLES = 5

json_schema = {
    "type": "object",
    "properties": {
        "text": {"type": "string"},
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "entity": {"type": "string"},
                    "types": {"type": "array", "items": {"type": "string"}}
                },
                "required": ["entity", "types"]
            }
        }
    },
    "required": ["text", "entities"]
}

results = []

for idx in range(NUM_EXAMPLES):
    example = examples[idx]
    text = " ".join(example['tokenized_text'])
    
    prompt = f"""Label the following text with named entities.

Entity Types (use ONLY these): {', '.join(entity_types)}

Text: {text}

Output Format:
{{
  "text": "{text}",
  "entities": [{{"entity": "entity max_name", "types": ["type"]}}]
}}

Generate ONLY valid JSON."""
    
    print(f"\nExample {idx+1}: {text}")
    print(f"\nPrompt:\n{prompt}\n")INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"

Example 1: the african queen

Prompt:
Label the following text with named entities.

Entity Types (use ONLY these): genre, year, plot, review, song, rating, character, trailer, director, title, actor

Text: the african queen

Output Format:
{
  "text": "the african queen",
  "entities": [{"entity": "entity max_name", "types": ["type"]}]
}

Generate ONLY valid JSON.

NORMAL RESPONSE:
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
We are labeling the text: "the african queen"
 The entity types we can use are: genre, year, plot, review, song, rating, character, trailer, director, title, actor

 The text "the african queen" is the title of a famous movie (1951 film directed by John Huston, starring Humphrey Bogart and Katharine Hepburn).

 Therefore, the most appropriate entity type for "the african queen" is "title".

 We note that:
   - It is not a genre (like comedy, drama, etc.)
   - It is not a year (it doesn't represent a year)
   - It is not a plot (it's too short to be a plot)
   - It is not a review (it's not a review text)
   - It is not a song (though there might be a song with the same name, in this context it's the movie title)
   - It is not a rating (like 5 stars)
   - It is not a character (though the boat is named "African Queen", the title refers to the movie)
   - It is not a trailer (it's the title, not a trailer)
   - It is not a director (it's not a person's name)
   - It is not an actor (it's not an actor's name)

 So, we assign the entity type "title".

 The output should be in JSON format as specified.


    
    # Normal
    print("NORMAL RESPONSE:")
    try:
        resp1 = client.chat.completions.create(
            model="qwen-3-235b-a22b-thinking-2507",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_completion_tokens=60000
        )
        print(resp1.choices[0].message.content)
        normal_data = (resp1.usage.prompt_tokens, resp1.usage.completion_tokens, 
                      len(resp1.choices[0].message.content),
                      'Yes' if resp1.choices[0].message.content.strip().startswith('{') else 'No',
                      'Yes' if '</think>' in resp1.choices[0].message.content else 'No')
    except Exception as e:
        print(f"ERROR: {e}")
        normal_data = (0, 0, 0, 'No', 'No')
    
    # Structured
    print("\nSTRUCTURED RESPONSE:")
    try:
        resp2 = client.chat.completions.create(
            model="qwen-3-235b-a22b-thinking-2507",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_completion_tokens=60000,
            response_format={
                "type": "json_schema",
                "json_schema": {"name": "ner_labeling", "schema": json_schema, "strict": True}
            }
        )
        print(resp2.choices[0].message.content)
        struct_data = (resp2.usage.prompt_tokens, resp2.usage.completion_tokens,
                      len(resp2.choices[0].message.content),
                      'Yes' if resp2.choices[0].message.content.strip().startswith('{') else 'No',
                      'Yes' if '</think>' in resp2.choices[0].message.content else 'No')
    except Exception as e:
        print(f"ERROR: {e}")
        struct_data = (0, 0, 0, 'No', 'No')
    
    results.append({
        'example': idx + 1,
        'input_tokens': f"N:{normal_data[0]} S:{struct_data[0]}",
        'output_tokens': f"N:{normal_data[1]} S:{struct_data[1]}",
        'total_length': f"N:{normal_data[2]} S:{struct_data[2]}",
        'valid_json': f"N:{normal_data[3]} S:{struct_data[3]}",
        'think_tag': f"N:{normal_data[4]} S:{struct_data[4]}"
    })
    print("\n" + "="*80)

print("\nSUMMARY:")
df = pd.DataFrame(results)
display(df)

INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"



Example 1: the african queen

Prompt:
Label the following text with named entities.

Entity Types (use ONLY these): genre, year, plot, review, song, rating, character, trailer, director, title, actor

Text: the african queen

Output Format:
{
  "text": "the african queen",
  "entities": [{"entity": "entity max_name", "types": ["type"]}]
}

Generate ONLY valid JSON.

NORMAL RESPONSE:


INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"


We are labeling the text: "the african queen"
 We have the following entity types: genre, year, plot, review, song, rating, character, trailer, director, title, actor

 The text "the african queen" is the title of a famous movie (released in 1951). Therefore, it should be labeled as a "title".

 Let's check:
 - It's not a genre (like comedy, drama, etc.)
 - It's not a year (it doesn't represent a year)
 - It's not a plot (it's too short and not descriptive of a story)
 - It's not a review (it's not a critical assessment)
 - It's not a song (though there might be a song with the same name, in this context it's the movie title)
 - It's not a rating (like 5 stars)
 - It's not a character (it's the title of the movie, not a character name)
 - It's not a trailer (it's the title, not a trailer)
 - It's not a director (it's not a person's name who directed)
 - It's not an actor (it's not an actor's name)

 Therefore, the only appropriate type is "title".

 We output in the specified JSON form

INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"


We are given a text: "is there a good movie that could be recommended which is funny"
 We have to label named entities with the following types: genre, year, plot, review, song, rating, character, trailer, director, title, actor

 Steps:
 1. Read the text and identify any named entities that fit the given types.
 2. The text is a question asking for a movie recommendation that is funny.

 Analysis:
 - The word "funny" is an adjective describing the type of movie. In the context of movies, "funny" typically refers to the genre (comedy). 
   However, note that the entity type "genre" is available. So we can label "funny" as a genre.

 But note: 
   The instruction says: "use ONLY these" types. We have to see if "funny" is a specific named entity of type genre? 
   Actually, "funny" is not a standard genre name (like "comedy"), but it is commonly used to mean comedy. 
   However, the problem says: "Label the following text with named entities". We are to extract entities that are present 

INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"


{"text": "is there a good movie that could be recommended which is funny", "entities": [{"entity": "good", "types": ["rating"]}, {"entity": "funny", "types": ["genre"]}]}


SUMMARY:


,example,input_tokens,output_tokens,total_length,valid_json,think_tag
0,1,N:91 S:265,N:407 S:381,N:1592 S:96,N:No S:Yes,N:Yes S:No
1,2,N:115 S:289,N:846 S:486,N:3371 S:137,N:No S:Yes,N:Yes S:No
2,3,N:103 S:277,N:753 S:393,N:3046 S:118,N:No S:Yes,N:Yes S:No
3,4,N:99 S:273,N:1441 S:3713,N:6048 S:163,N:No S:Yes,N:Yes S:No
4,5,N:109 S:283,N:1306 S:1519,N:5279 S:170,N:No S:Yes,N:Yes S:No
